In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import time
import numpy as np
import torch

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux

# Kept lightweight on purpose -- this is meant to settle "which is faster",
# not to be a heavy sweep.
N_CALLS = 2000

drive = ieee_machine2_trained_neural_flux()
flux_model = drive.flux  # the NeuralFlux instance (numpy-analytic)
net = flux_model.net
scaler = flux_model.scaler
device = flux_model.device

rng = np.random.default_rng(0)
omegas = rng.uniform(0, 300, size=N_CALLS)
currs = rng.uniform(-20, 20, size=(N_CALLS, 4))

print(f"N_CALLS = {N_CALLS}, hidden_sizes = {net.hidden_sizes}, activation = {net.activation}")

In [2]:
def clear_cache():
    flux_model._flux_cache = None
    flux_model._inductance_cache = None

clear_cache()
t0 = time.perf_counter()
for k in range(N_CALLS):
    _ = flux_model.flux(omegas[k], currs[k])
    _ = flux_model.inductance(omegas[k], currs[k])
t_numpy = time.perf_counter() - t0

print(f"numpy analytic: {t_numpy:.4f}s total  ({1e6*t_numpy/N_CALLS:.2f} us/call)")

numpy analytic: 0.0424s total  (21.18 us/call)


In [3]:
mean_t = torch.from_numpy(scaler.mean_).float().to(device)
scale_t = torch.from_numpy(scaler.scale_).float().to(device)
flux_pm_t = torch.from_numpy(np.asarray(flux_model.flux_pm)).float().to(device)
L_stat_t = torch.from_numpy(np.asarray(flux_model.L_stat)).float().to(device)


def torch_flux_and_inductance(omega, curr_dq):
    x = torch.tensor([omega, *curr_dq], dtype=torch.float32, device=device, requires_grad=True)
    x_normed = (x - mean_t) / scale_t
    w_res = net(x_normed.unsqueeze(0)).squeeze()
    grad = torch.autograd.grad(w_res, x, create_graph=True)[0]
    d_wres_d_i = grad[1:]
    flux = flux_pm_t + L_stat_t @ x[1:].detach() + d_wres_d_i
    hess_rows = []
    for j in range(4):
        row = torch.autograd.grad(grad[1 + j], x, retain_graph=True)[0]
        hess_rows.append(row[1:])
    hess = torch.stack(hess_rows)
    inductance = L_stat_t + hess
    return flux.detach().numpy(), inductance.detach().numpy()


t0 = time.perf_counter()
for k in range(N_CALLS):
    _ = torch_flux_and_inductance(omegas[k], currs[k])
t_torch = time.perf_counter() - t0

print(f"torch autograd: {t_torch:.4f}s total  ({1e6*t_torch/N_CALLS:.2f} us/call)")

torch autograd: 3.8677s total  (1933.85 us/call)


In [4]:
clear_cache()
max_flux_diff = 0.0
max_ind_diff = 0.0
for k in range(10):
    f_np = flux_model.flux(omegas[k], currs[k])
    l_np = flux_model.inductance(omegas[k], currs[k])
    f_t, l_t = torch_flux_and_inductance(omegas[k], currs[k])
    max_flux_diff = max(max_flux_diff, np.max(np.abs(f_np - f_t)))
    max_ind_diff = max(max_ind_diff, np.max(np.abs(l_np - l_t)))

print(f"max |flux_numpy - flux_torch|:        {max_flux_diff:.2e}")
print(f"max |inductance_numpy - inductance_torch|: {max_ind_diff:.2e}")

max |flux_numpy - flux_torch|:        1.02e-09
max |inductance_numpy - inductance_torch|: 4.81e-11


In [5]:
print(f"N_CALLS = {N_CALLS}")
print(f"numpy analytic:        {t_numpy:.4f}s  ({1e6*t_numpy/N_CALLS:.2f} us/call)")
print(f"torch autograd:        {t_torch:.4f}s  ({1e6*t_torch/N_CALLS:.2f} us/call)")
print(f"speedup (torch/numpy): {t_torch/t_numpy:.2f}x")

N_CALLS = 2000
numpy analytic:        0.0424s  (21.18 us/call)
torch autograd:        3.8677s  (1933.85 us/call)
speedup (torch/numpy): 91.33x
